# The Quantum Fourier Transform

## The QFT is the quantum analogue of the discrete Fourier transform. It is the central subroutine in Shor's algorithm, quantum phase estimation, and a long list of other quantum algorithms. Crucially, the QFT on $n$ qubits runs in $O(n^2)$ gates — exponentially faster than the classical DFT on $2^n$ amplitudes.

# 1. Definition

## On the basis state $|x\rangle$ with $x \in \{0, 1, \dots, N-1\}$ and $N = 2^n$:

$$ \Large  \mathrm{QFT}\,|x\rangle = \frac{1}{\sqrt N} \sum_{y=0}^{N-1} e^{2\pi i\, xy / N}\, |y\rangle. $$

## By linearity this defines its action on arbitrary states. Note: this is exactly the DFT matrix, viewed as a unitary.

In [1]:
import numpy as np

def qft_matrix(n):
    N = 2**n
    omega = np.exp(2j*np.pi/N)
    return np.array([[omega**(x*y) for y in range(N)] for x in range(N)]) / np.sqrt(N)

F = qft_matrix(3)
print('QFT unitary?', np.allclose(F.conj().T @ F, np.eye(8)))

QFT unitary? True


# 2. Product form

## Write $y = y_1 y_2 \dots y_n$ in binary. After some algebra, the QFT can be rewritten as a *tensor product*:

$$ \Large  \mathrm{QFT}\,|x\rangle = \bigotimes_{k=1}^{n} \frac{1}{\sqrt 2}\big(|0\rangle + e^{2\pi i\, x / 2^k}|1\rangle\big). $$

## This product form is what makes the QFT efficiently implementable: each output qubit only needs a single-qubit rotation conditioned on the input bits.

# 3. Efficient circuit

## The standard QFT circuit on $n$ qubits:

## 1. For $k$ from $1$ to $n$:
   - Apply $H$ to qubit $k$.
   - For each $j > k$, apply a controlled $R_{j-k+1}$ from qubit $j$ to qubit $k$, where $R_m = \text{diag}(1, e^{2\pi i / 2^m})$.
## 2. Reverse the order of the qubits (a series of SWAPs).

## Total: $O(n^2)$ gates. Compare with the classical FFT, which uses $O(N \log N) = O(2^n n)$ operations on $N$ amplitudes.

In [2]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator
import numpy as np

def qft_circuit(n):
    qc = QuantumCircuit(n)
    for k in range(n):
        qc.h(k)
        for j in range(k+1, n):
            qc.cp(np.pi / (2 ** (j - k)), j, k)
    # reverse qubit order
    for k in range(n // 2):
        qc.swap(k, n - 1 - k)
    return qc

n = 3
qc = qft_circuit(n)
U  = Operator(qc).data
F  = qft_matrix(n)
print('Qiskit QFT matches definition?', np.allclose(U, F))

Qiskit QFT matches definition? False


# 4. Inverse QFT

## $\mathrm{QFT}^{-1}$ is the conjugate-transpose: the same circuit with reversed gate order and conjugated phases. Most algorithms (phase estimation, Shor) use the inverse QFT to *read out* a phase encoded in the state.

# 5. Why it matters

## - **Period finding**: the QFT extracts the period of a periodic function — the core trick of Shor.
## - **Phase estimation** (notebook 2): the QFT^{-1} converts a quantum-encoded phase into a measurable bitstring.
## - **HHL** for linear systems uses phase estimation, hence the QFT.
## - Many simulation algorithms use the QFT to swap between position and momentum representations.

## Of all quantum subroutines, the QFT is the one to remember.

# Recap

## - $\mathrm{QFT}|x\rangle = \frac{1}{\sqrt N}\sum_y e^{2\pi i xy/N}|y\rangle$.
## - Tensor-product form $\Rightarrow$ $O(n^2)$-gate circuit.
## - $\mathrm{QFT}^{-1}$ is used as a measurement basis change in many algorithms.